In [1]:
import pandas as pd
import numpy as np
import re
import wrds

### 1) iShares AGG ETF -> get cusips

In [2]:
# reading latest CSV file from the iShares AGG ETF website
import re
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

PAGE_URL = "https://www.ishares.com/us/products/239458/ishares-core-total-us-bond-market-etf"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}
LINK_TEXT = re.compile(
    r"detailed holdings and analytics|download latest fund holdings", re.I
)


def get_project_root(marker: str = "pyproject.toml") -> Path:
    """Walk up from the current directory until the folder containing `marker` is found."""
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / marker).exists():
            return folder
    raise FileNotFoundError(f"Couldn't find {marker} in {here} or any parent folder.")


def find_holdings_link(soup: BeautifulSoup, base_url: str) -> str:
    """Return the absolute URL of the holdings CSV link."""
    anchors = soup.find_all("a", href=True)

    # 1) Match on the visible link text (old and new wording)
    for a in anchors:
        if LINK_TEXT.search(a.get_text(" ", strip=True)):
            return urljoin(base_url, a["href"])

    # 2) Fallback: any href that looks like a holdings CSV
    for a in anchors:
        href = a["href"].lower()
        if "holdings" in href and ("csv" in href or "filetype=csv" in href):
            return urljoin(base_url, a["href"])

    raise LookupError("Couldn't find a holdings CSV link on the page.")


def download_agg_holdings(
    page_url: str = PAGE_URL,
    out_dir: Path | str | None = None,
    filename: str = "latest_AGG_holdings.CSV",
) -> Path:
    """Download the AGG holdings CSV to <project root>/data/<filename>, overwriting if present."""
    out_dir = Path(out_dir) if out_dir else get_project_root() / "data"
    out_path = out_dir / filename
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with requests.Session() as s:
        s.headers.update(HEADERS)

        page = s.get(page_url, timeout=30)
        page.raise_for_status()
        soup = BeautifulSoup(page.text, "html.parser")

        # page.url is the post-redirect URL, so relative hrefs resolve correctly
        csv_url = find_holdings_link(soup, page.url)

        resp = s.get(csv_url, headers={"Referer": page.url}, timeout=60)
        resp.raise_for_status()

        if "html" in resp.headers.get("Content-Type", "").lower():
            raise ValueError(
                f"Expected a CSV but got HTML from {csv_url} (possibly blocked)."
            )

        out_path.write_bytes(resp.content)  # "wb" semantics -> replaces existing file

    print(f"Saved {out_path} ({out_path.stat().st_size:,} bytes)")
    return out_path

path = download_agg_holdings()

Saved /Users/rosana/CODE/Capstone/data/latest_AGG_holdings.CSV (3,704,795 bytes)


In [3]:
# cleaning the raw data from iShares

csv_path = get_project_root() / "data" / "latest_AGG_holdings.CSV"
ishares_df = pd.read_csv(
    csv_path,
    skiprows=9,
    thousands=",",
    encoding="utf-8-sig",
    dtype={"CUSIP": "string"},   # read as text so leading zeros are never lost
)

ishares_df["CUSIP"] = ishares_df["CUSIP"].str.strip().str.zfill(9)

/var/folders/_3/qz73tcnx6png_73ny7dx1yjc0000gn/T/ipykernel_6624/2214524420.py:12: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  ishares_df["CUSIP"] = ishares_df["CUSIP"].str.strip().str.zfill(9)


In [4]:
ishares_df.columns

Index(['Name', 'Sector', 'Asset Class', 'Market Value', 'Weight (%)',
       'Notional Value', 'Par Value', 'CUSIP', 'ISIN', 'SEDOL', 'Location',
       'Exchange', 'Currency', 'Duration', 'YTM (%)', 'FX Rate', 'Maturity',
       'Coupon (%)', 'Mod. Duration', 'Yield to Call (%)',
       'Yield to Worst (%)', 'Real Duration', 'Real YTM (%)',
       'Market Currency', 'Accrual Date', 'Effective Date'],
      dtype='object')

In [5]:
# get all cusips from ishares_df which will be queried in the WRDS db
ishares_cusips = ishares_df["CUSIP"].dropna().unique()
ishares_cusips_list = ishares_cusips.tolist()
ishares_cusips_sql = ", ".join(f"'{c}'" for c in ishares_cusips_list)

In [6]:
print(f"number of unique iShares cusips: {len(ishares_cusips):,}")

number of unique iShares cusips: 13,432


### 2a) WRDS -> get Corps data from BondRets

In [7]:
db = wrds.Connection(wrds_username="rosanagao")

Loading library list...
Done


In [8]:
db.list_libraries() # wrdsapps_bondret

['aha_sample',
 'ahasamp',
 'altrata_exec_samp',
 'altsamp',
 'audit',
 'audit_acct_os',
 'audit_audit_comp',
 'audit_common',
 'audit_corp_legal',
 'auditsmp',
 'auditsmp_all',
 'bank',
 'bank_all',
 'bank_premium_samp',
 'banksamp',
 'block',
 'block_all',
 'boardex',
 'boardex_na',
 'boardex_trial',
 'boardsmp',
 'bvd_amadeus_trial',
 'bvd_bvdbankf_trial',
 'bvd_orbis_trial',
 'bvdsamp',
 'calcbench_trial',
 'calcbnch',
 'candid_samp',
 'cboe',
 'cboe_all',
 'cboe_sample',
 'cboesamp',
 'cddsamp',
 'ciq',
 'ciq_common',
 'ciq_pplintel',
 'ciq_transcripts',
 'ciqsamp',
 'ciqsamp_capstrct',
 'ciqsamp_common',
 'ciqsamp_keydev',
 'ciqsamp_pplintel',
 'ciqsamp_ratings',
 'ciqsamp_transactions',
 'ciqsamp_transcripts',
 'cisdmsmp',
 'columnar',
 'comp',
 'comp_bank_daily',
 'comp_execucomp',
 'comp_na_daily_all',
 'comp_pit',
 'comp_segments_hist_daily',
 'comp_snapshot',
 'comph',
 'compsamp',
 'compsamp_all',
 'compsamp_computext',
 'compsamp_snapshot',
 'compseg',
 'compsnap',
 'contr

In [9]:
# relevant WRDS libraries & tables
libraries = db.list_libraries() # wrdsapps_bondret
bondret_tables = db.list_tables(library="wrdsapps_bondret")

In [10]:
db.describe_table(library="wrdsapps_bondret", table="trace_enhanced_clean")

Approximately 334769344 rows in wrdsapps_bondret.trace_enhanced_clean.


,name,nullable,type,comment
0,cusip_id,True,VARCHAR(9),CUSIP ID
1,bond_sym_id,True,VARCHAR(15),TRACE Bond Symbol
2,company_symbol,True,VARCHAR(6),WRDS Generated Company Symbol (issuer stock ti...
3,trd_exctn_dt,True,DATE,Execution Date
4,trd_exctn_tm,True,DOUBLE PRECISION,Execution Time
5,trd_rpt_dt,True,DATE,Report Date
6,trd_rpt_tm,True,DOUBLE PRECISION,Report Time
7,msg_seq_nb,True,VARCHAR(7),Message Sequence Number
8,trc_st,True,VARCHAR(1),Trade Status
9,scrty_type_cd,True,VARCHAR(5),Security Type


In [11]:
db.connection.rollback()
corps_query = f"""
    SELECT *
    FROM wrdsapps_bondret.bondret
    WHERE cusip IN ({ishares_cusips_sql})
    AND date >= '2025-01-01'
"""
wrds_corps_df = db.raw_sql(corps_query)

In [12]:
print(f"number of unique cusips in wrds_corps_df: {wrds_corps_df['cusip'].nunique():,}")
wrds_corps_df.columns

number of unique cusips in wrds_corps_df: 6,710


Index(['date', 'issue_id', 'cusip', 'bond_sym_id', 'bsym', 'isin',
       'company_symbol', 'bond_type', 'security_level', 'conv',
       'offering_date', 'offering_amt', 'offering_price', 'principal_amt',
       'maturity', 'treasury_maturity', 'coupon', 'day_count_basis',
       'dated_date', 'first_interest_date', 'last_interest_date', 'ncoups',
       'amount_outstanding', 'r_sp', 'r_mr', 'r_fr', 'n_sp', 'n_mr', 'n_fr',
       'rating_num', 'rating_cat', 'rating_class', 't_date', 't_volume',
       't_dvolume', 't_spread', 't_yld_pt', 'yield', 'price_eom_flg',
       'price_eom', 'price_ldm', 'price_l5m', 'gap', 'coupmonth', 'nextcoup',
       'coupamt', 'coupacc', 'multicoups', 'ret_eom', 'ret_ldm', 'ret_l5m',
       'tmt', 'remcoups', 'duration', 'defaulted', 'default_date',
       'default_type', 'reinstated', 'reinstated_date'],
      dtype='object')

### 2b) get treasuries from CRSP

In [13]:
# count how many treasuries are in the iShares AGG ETF - cusip starts with 912%
num_treasuries = ishares_df["CUSIP"].str.startswith("912").sum()
print(f"number of treasuries in iShares AGG ETF: {num_treasuries:,}")

number of treasuries in iShares AGG ETF: 297


In [14]:
db.connection.rollback()
#db.list_tables(library="trace_enhanced") 
db.describe_table(library="crsp", table="tfz_iss")

Approximately 8359 rows in crsp.tfz_iss.


,name,nullable,type,comment
0,kytreasno,True,DOUBLE PRECISION,KYTREASNO
1,kycrspid,True,VARCHAR(15),KYCRSPID
2,crspid,True,VARCHAR(15),CRSPID
3,tcusip,True,VARCHAR(8),TCUSIP
4,tdatdt,True,DATE,TDATDT
5,tmatdt,True,DATE,TMATDT
6,iwhy,True,DOUBLE PRECISION,IWHY
7,tcouprt,True,"NUMERIC(6, 3)",TCOUPRT
8,tnippy,True,DOUBLE PRECISION,TNIPPY
9,tvalfc,True,DOUBLE PRECISION,TVALFC


In [15]:
cusips8 = sorted({c[:8] for c in ishares_cusips_list})

db.connection.rollback()
wrds_treasuries_df = db.raw_sql(
    """
    SELECT kytreasno, crspid, tcusip, tdatdt, tmatdt, tcouprt
    FROM crsp.tfz_iss
    WHERE tcusip IN %(cusips)s
    """,
    params={"cusips": tuple(cusips8)},
)
print(f"{len(cusips8)} CUSIPs requested, {len(wrds_treasuries_df)} matched")

13432 CUSIPs requested, 253 matched


In [16]:
print("number of unique cusips in wrds_treasuries_df: {:,}".format(wrds_treasuries_df["tcusip"].nunique()))

number of unique cusips in wrds_treasuries_df: 253


In [17]:
wrds_treasuries_df.head(10)

,kytreasno,crspid,tcusip,tdatdt,tmatdt,tcouprt
0,204092.0,20280815.105500,912810FE,1998-08-15,2028-08-15,5.5
1,204093.0,20281115.105250,912810FF,1998-11-15,2028-11-15,5.25
2,204094.0,20290215.105250,912810FG,1999-02-15,2029-02-15,5.25
3,204095.0,20290815.106120,912810FJ,1999-08-15,2029-08-15,6.125
4,204096.0,20300515.106250,912810FM,1999-11-15,2030-05-15,6.25
5,204098.0,20360215.104500,912810FT,2006-02-15,2036-02-15,4.5
6,204099.0,20370215.104750,912810PT,2007-02-15,2037-02-15,4.75
7,204100.0,20370515.105000,912810PU,2007-05-15,2037-05-15,5.0
8,204101.0,20380215.104370,912810PW,2008-02-15,2038-02-15,4.375
9,204102.0,20380515.104500,912810PX,2008-05-15,2038-05-15,4.5


### 2c) Get ABS/MBS from TRACE?

In [18]:
# get all cusips that were not matched in the bondrets or tfz_iss tables
matched_cusips = set(wrds_corps_df["cusip"].unique()).union(set(wrds_treasuries_df["tcusip"].unique()))
unmatched_cusips = set(ishares_cusips_list) - matched_cusips
print(f"number of unmatched cusips: {len(unmatched_cusips):,}")

number of unmatched cusips: 6,722


In [19]:
db.connection.rollback()
db.list_tables(library="trace") 
db.describe_table(library="trace", table="absmasterfile")

Approximately 776595 rows in trace.absmasterfile.


,name,nullable,type,comment
0,cusip_id,True,VARCHAR,CUSIP
1,sym_cd,True,VARCHAR,"Issue Symbol, also called bond_sym_id in the T..."
2,bsym_id,True,VARCHAR,WRDS-Generated Company Symbol (Stock Ticker)
3,company_symbol,True,VARCHAR,Company Symbol
4,mstr_deal_id,True,VARCHAR,Master Deal ID
5,issuer_nm,True,VARCHAR,Issuer Name
6,trnch_nb,True,VARCHAR,Tranche Number
7,scrty_ds,True,VARCHAR,Security Description
8,sub_prdct_type,True,VARCHAR,Sub-Product Type Code
9,scrty_sbtp_cd,True,VARCHAR,Sub-Product Asset Code


In [20]:
unmatched_cusips_sql = ", ".join(f"'{c}'" for c in unmatched_cusips)
trace_abs_query = f"""
    SELECT *
    FROM trace_enhanced.absmasterfile
    WHERE cusip_id IN ({unmatched_cusips_sql})
""" 
abs_df = db.raw_sql(trace_abs_query)
print(f"number of unmatched cusips found in TRACE ABS Masterfile: {len(abs_df):,}")


number of unmatched cusips found in TRACE ABS Masterfile: 494


In [21]:
trace_mbs_query = f"""
    SELECT *
    FROM trace_enhanced.mbsmasterfile
    WHERE cusip_id IN ({unmatched_cusips_sql})
"""
mbs_df = db.raw_sql(trace_mbs_query)
print(f"number of unmatched cusips found in TRACE MBS Masterfile: {len(mbs_df):,}")

number of unmatched cusips found in TRACE MBS Masterfile: 11,576


### 3. Cleaning all the dfs
To Do:
- clean up dates
- get relevant columns for all dfs
- merge all datasets onto the ishares df
- figure out which cusips are still missing

In [22]:
# date cleaning for all dataframes
ishares_df
ishares_columns = ['CUSIP', 'Name', 'Sector', 'Asset Class', 'Market Value', 'Weight (%)', 'Notional Value', 'Par Value', 'Currency', \
           'Maturity', 'Coupon (%)', 'Mod. Duration', 'Duration', 'YTM (%)', 'Accrual Date', 'Effective Date']
ishares_df = ishares_df[ishares_columns]
print(f"total number of securities in ishares df: {ishares_df['CUSIP'].nunique()}")

total number of securities in ishares df: 13432


In [23]:
wrds_corps_df
corps_columns = ['cusip', 'date', 'bond_type', 'security_level', 'offering_date', 'maturity', 'dated_date', \
                 'principal_amt', 'coupon', 'day_count_basis', 'ncoups', 'rating_num', 'rating_cat', 'rating_class', 'duration', \
                    'defaulted', 'default_date', 'default_type', 'reinstated', 'reinstated_date']
wrds_corps_df = wrds_corps_df[corps_columns]
# TODO: drop dupe cols based on valuation date!
print(f"number of corps: {wrds_corps_df['cusip'].nunique()}")

number of corps: 6710


In [24]:
wrds_treasuries_df
treasuries_columns = ['tcusip', 'tmatdt', 'tcouprt', 'tdatdt'] # cusip, maturity, coupon rate, issue date
wrds_treasuries_df[treasuries_columns]
# TODO: convert 8char cusip back to 9char
print(f"number of treasuries: {wrds_treasuries_df['tcusip'].nunique()}")

number of treasuries: 253


In [25]:
abs_df
abs_columns = ['cusip_id', 'issuer_nm', 'scrty_ds', 'cpn_rt', 'cpn_type_cd', 'mtrty_dt', 'grade', 'stdt', 'enddt']
abs_df = abs_df[abs_columns]
# TODO: figure out how to clean up dupe cols
print(f"number of abs: {abs_df['cusip_id'].nunique()}")


number of abs: 134


In [26]:
mbs_df.columns
mbs_columns = ['cusip_id', 'issuer_nm', 'scrty_ds', 'cpn_rt', 'cpn_type_cd', 'mtrty_dt', 'grade', 'issue_dt', 'stdt', 'enddt']
mbs_df[mbs_columns]
# TODO: get rid of dupes here too
print(f"number of unique mbs: {mbs_df['cusip_id'].nunique()}")

number of unique mbs: 2904


In [ ]:
# standard functions for data cleaning
def normalize_cusip(series):
    """Normalize CUSIP strings without losing leading zeros."""
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"[\s-]", "", regex=True)
        .replace("", pd.NA)
    )


def parse_date(series):
    """Parse dates, converting invalid or missing values to NaT."""
    return pd.to_datetime(series, errors="coerce")


def to_numeric(series):
    """Convert numeric columns, including strings with commas."""
    return pd.to_numeric(
        series.astype("string").str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False).str.strip(),
        errors="coerce"
    )


def standardize_columns(df, mapping):
    """Rename only columns that actually exist in the input."""
    return df.rename(
        columns={k: v for k, v in mapping.items() if k in df.columns}
    ).copy()


def assert_unique_cusip(df, name):
    """Fail loudly if a supposedly unique security table has duplicates."""
    duplicated = df.loc[df["CUSIP"].duplicated(keep=False), "CUSIP"]
    if not duplicated.empty:
        raise ValueError(
            f"{name} contains duplicate CUSIPs: "
            f"{duplicated.dropna().unique()[:10].tolist()}"
        )


def latest_record_asof(df, cusip_col, date_col, valuation_date):
    """
    Retain the latest observation per CUSIP on or before valuation_date.
    Assumes date_col is an observation/record date, not a validity end date.
    """
    out = df.copy()
    out[cusip_col] = normalize_cusip(out[cusip_col])
    out[date_col] = parse_date(out[date_col])
    valuation_date = pd.Timestamp(valuation_date)

    out = out.dropna(subset=[cusip_col])
    out = out.loc[out[date_col].le(valuation_date)]
    out = out.sort_values([cusip_col, date_col], kind="stable")
    return out.drop_duplicates(cusip_col, keep="last")


: 

In [ ]:
# clean iShares df

def clean_ishares(ishares_df):
    df = ishares_df.copy()
    df.columns = df.columns.str.strip()

    df["CUSIP"] = normalize_cusip(df["CUSIP"])

    numeric_cols = [
        "Market Value", "Weight (%)", "Notional Value", "Par Value",
        "Duration", "YTM (%)", "FX Rate", "Coupon (%)",
        "Mod. Duration", "Yield to Call (%)", "Yield to Worst (%)",
        "Real Duration", "Real YTM (%)"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = to_numeric(df[col])

    for col in ["Maturity", "Accrual Date", "Effective Date"]:
        if col in df.columns:
            df[col] = parse_date(df[col])

    df["index_weight"] = df["Weight (%)"] / 100.0
    df["coupon_rate_pct"] = df["Coupon (%)"]

    df = df.dropna(subset=["CUSIP"])

    # The index snapshot should ordinarily have one row per CUSIP.
    # Do not silently combine conflicting security records.
    duplicates = df.loc[df["CUSIP"].duplicated(keep=False)]
    if not duplicates.empty:
        print(
            "Warning: duplicate iShares CUSIPs found. "
            "Inspect these rows before choosing a record:\n",
            duplicates[["CUSIP", "Name", "Weight (%)"]].head(20)
        )
        # Retain the first occurrence provisionally; replace this rule
        # with explicit reconciliation if duplicate records are meaningful.
        df = df.drop_duplicates("CUSIP", keep="first")

    return df.reset_index(drop=True)


ishares_clean = clean_ishares(ishares_df)
ishares_clean

In [ ]:

def clean_corporates(wrds_corps_df, valuation_date):
    df = wrds_corps_df.copy()
    df.columns = df.columns.str.strip()

    df["CUSIP"] = normalize_cusip(df["cusip"])
    df["record_date"] = parse_date(df["date"])
    df["maturity_date_wrds"] = parse_date(df["maturity"])
    df["offering_date_wrds"] = parse_date(df["offering_date"])

    numeric_cols = [
        "coupon", "duration", "offering_amt", "principal_amt",
        "amount_outstanding", "rating_num", "yield"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = to_numeric(df[col])

    # Exclude future observations and choose the latest record as of
    # the valuation date.
    df = latest_record_asof(
        df, "CUSIP", "record_date", valuation_date
    )

    mapping = {
        "issue_id": "corp_issue_id",
        "bond_type": "corp_bond_type",
        "security_level": "corp_security_level",
        "conv": "corp_convertible_flag",
        "maturity_date_wrds": "corp_maturity_date",
        "offering_date_wrds": "corp_offering_date",
        "coupon": "corp_coupon_pct",
        "day_count_basis": "corp_day_count_basis",
        "offering_amt": "corp_offering_amount",
        "principal_amt": "corp_principal_amount",
        "amount_outstanding": "corp_amount_outstanding",
        "rating_num": "corp_rating_num",
        "rating_cat": "corp_rating_category",
        "rating_class": "corp_rating_class",
        "duration": "corp_duration",
        "defaulted": "corp_defaulted",
        "default_date": "corp_default_date",
        "reinstated": "corp_reinstated",
        "isin": "corp_isin",
        "record_date": "corp_record_date",
    }

    keep = [
        "CUSIP", "issue_id", "bond_type", "security_level", "conv",
        "maturity_date_wrds", "offering_date_wrds", "coupon",
        "day_count_basis", "offering_amt", "principal_amt",
        "amount_outstanding", "rating_num", "rating_cat", "rating_class",
        "duration", "defaulted", "default_date", "reinstated", "isin",
        "record_date"
    ]
    df = df[[c for c in keep if c in df.columns]]
    df = standardize_columns(df, mapping)

    # Normalize output dates and numeric types.
    for col in ["corp_maturity_date", "corp_offering_date",
                "corp_record_date", "corp_default_date"]:
        if col in df.columns:
            df[col] = parse_date(df[col])

    assert_unique_cusip(df, "Corporate bonds")
    return df.reset_index(drop=True)


corps_clean = clean_corporates(
    wrds_corps_df,
    valuation_date="2026-10-20"  # replace with your actual snapshot date
)


In [ ]:

def cusip_check_digit(stem):
    """
    Calculate the standard CUSIP check digit for an 8-character stem.
    Returns a 9-character CUSIP, or None for invalid input.
    """
    if not isinstance(stem, str) or len(stem) != 8:
        return None

    values = []
    for ch in stem.upper():
        if ch.isdigit():
            values.append(int(ch))
        elif "A" <= ch <= "Z":
            values.append(ord(ch) - ord("A") + 10)
        elif ch == "*":
            values.append(36)
        elif ch == "@":
            values.append(37)
        elif ch == "#":
            values.append(38)
        else:
            return None

    total = 0
    for i, value in enumerate(values):
        product = value * (2 if i % 2 == 1 else 1)
        total += product // 10 + product % 10

    return stem.upper() + str((10 - total % 10) % 10)


def clean_treasuries(wrds_treasuries_df):
    df = wrds_treasuries_df.copy()
    df.columns = df.columns.str.strip()

    df["cusip_stem"] = normalize_cusip(df["tcusip"])
    df["CUSIP"] = df["cusip_stem"].map(cusip_check_digit)

    df["treasury_issue_date"] = parse_date(df["tdatdt"])
    df["treasury_maturity_date"] = parse_date(df["tmatdt"])
    df["treasury_coupon_pct"] = to_numeric(df["tcouprt"])

    df = df.dropna(subset=["CUSIP"])

    keep = [
        "CUSIP", "cusip_stem", "kytreasno", "crspid",
        "treasury_issue_date", "treasury_maturity_date",
        "treasury_coupon_pct"
    ]
    df = df[keep]

    # Multiple Treasury records per CUSIP are possible.
    # Collapse only when the resulting reference fields agree.
    # Otherwise, investigate instead of silently discarding conflicts.
    reference_cols = [
        "treasury_maturity_date", "treasury_coupon_pct"
    ]

    conflicts = (
        df.groupby("CUSIP")[reference_cols]
        .nunique(dropna=True)
        .gt(1).any(axis=1)
    )
    if conflicts.any():
        raise ValueError(
            "Conflicting Treasury reference data for CUSIPs: "
            f"{conflicts[conflicts].index.tolist()[:10]}"
        )

    df = (
        df.sort_values("treasury_issue_date", kind="stable")
        .drop_duplicates("CUSIP", keep="last")
    )

    assert_unique_cusip(df, "Treasuries")
    return df.reset_index(drop=True)


treasuries_clean = clean_treasuries(wrds_treasuries_df)


In [ ]:

def clean_structured_bonds(df, kind, valuation_date):
    """
    Clean ABS/MBS reference data.
    Prefer records valid on valuation_date; if none are valid for a
    CUSIP, use the most recent record starting before that date.
    """
    df = df.copy()
    df.columns = df.columns.str.strip()

    df["CUSIP"] = normalize_cusip(df["cusip_id"])
    df["record_start"] = parse_date(df["stdt"])

    if "enddt" in df.columns:
        df["record_end"] = parse_date(df["enddt"])
    else:
        df["record_end"] = pd.NaT

    df["maturity_date"] = parse_date(df["mtrty_dt"])
    df["coupon_pct"] = to_numeric(df["cpn_rt"])
    df["issue_date"] = (
        parse_date(df["issue_dt"])
        if "issue_dt" in df.columns
        else pd.NaT
    )

    valuation_date = pd.Timestamp(valuation_date)
    df = df.dropna(subset=["CUSIP"])

    # Only records that began by the valuation date.
    df = df.loc[
        df["record_start"].isna()
        | df["record_start"].le(valuation_date)
    ].copy()

    # A record with no end date is treated as open-ended.
    active = (
        df["record_end"].isna()
        | df["record_end"].ge(valuation_date)
    )

    df["_active"] = active.astype(int)
    df = df.sort_values(
        ["CUSIP", "_active", "record_start"],
        kind="stable",
        na_position="first"
    )

    # Prefer an active record, then the latest applicable start date.
    df = df.drop_duplicates("CUSIP", keep="last")

    prefix = kind.lower()
    mapping = {
        "issuer_nm": f"{prefix}_issuer",
        "scrty_ds": f"{prefix}_description",
        "sub_prdct_type": f"{prefix}_product_type",
        "scrty_sbtp_cd": f"{prefix}_security_subtype",
        "cpn_type_cd": f"{prefix}_coupon_type",
        "intrs_type_cd": f"{prefix}_interest_type",
        "cpn_rt": f"{prefix}_coupon_pct",
        "mtrty_dt": f"{prefix}_maturity_date",
        "maturity_date": f"{prefix}_maturity_date",
        "coupon_pct": f"{prefix}_coupon_pct",
        "grade": f"{prefix}_grade",
        "mstr_deal_id": f"{prefix}_deal_id",
        "trnch_nb": f"{prefix}_tranche_number",
        "trnch_id": f"{prefix}_tranche_id",
        "pool_nb": f"{prefix}_pool_number",
        "issue_dt": f"{prefix}_issue_date",
        "issue_date": f"{prefix}_issue_date",
        "record_start": f"{prefix}_record_start",
        "record_end": f"{prefix}_record_end",
        "tba_stlmt_cd": f"{prefix}_tba_settlement_code",
        "ind_144a": f"{prefix}_rule_144a_flag",
    }

    keep = [
        "CUSIP", "issuer_nm", "scrty_ds", "sub_prdct_type",
        "scrty_sbtp_cd", "cpn_type_cd", "intrs_type_cd", "cpn_rt",
        "mtrty_dt", "maturity_date", "coupon_pct", "grade",
        "mstr_deal_id", "trnch_nb", "trnch_id", "pool_nb",
        "issue_dt", "issue_date", "record_start", "record_end",
        "tba_stlmt_cd", "ind_144a"
    ]
    df = df[[c for c in keep if c in df.columns]]
    df = standardize_columns(df, mapping)

    assert_unique_cusip(df, kind)
    return df.reset_index(drop=True)


abs_clean = clean_structured_bonds(
    abs_df, "ABS", "2026-10-20"
)

mbs_clean = clean_structured_bonds(
    mbs_df, "MBS", "2026-10-20"
)


In [ ]:

def merge_bond_reference_data(
    ishares_clean,
    corps_clean,
    treasuries_clean,
    abs_clean,
    mbs_clean,
):
    base = ishares_clean.copy()
    base["CUSIP"] = normalize_cusip(base["CUSIP"])
    assert_unique_cusip(base, "iShares base universe")

    # Each auxiliary table must have at most one row per CUSIP.
    sources = {
        "corporates": corps_clean,
        "treasuries": treasuries_clean,
        "ABS": abs_clean,
        "MBS": mbs_clean,
    }

    for name, source in sources.items():
        assert_unique_cusip(source, name)

    n_before = len(base)

    merged = base.copy()
    for source in sources.values():
        merged = merged.merge(
            source,
            on="CUSIP",
            how="left",
            validate="one_to_one",
        )

    assert len(merged) == n_before, (
        "Merge changed the number of iShares securities."
    )
    assert merged["CUSIP"].is_unique

    # Coverage diagnostics: unmatched securities are not necessarily
    # errors; they may belong to another instrument type.
    diagnostics = pd.DataFrame({
        "source": list(sources.keys()),
        "matched_securities": [
            int(merged[
                merged.columns.intersection(
                    [c for c in source.columns if c != "CUSIP"]
                )
            ].notna().any(axis=1).sum())
            if any(c != "CUSIP" for c in source.columns)
            else 0
            for source in sources.values()
        ],
    })

    return merged, diagnostics


bond_reference_df, join_diagnostics = merge_bond_reference_data(
    ishares_clean,
    corps_clean,
    treasuries_clean,
    abs_clean,
    mbs_clean,
)

print("Merged shape:", bond_reference_df.shape)
print(join_diagnostics)
print(bond_reference_df.head())

In [ ]:

def report_match_rates(merged):
    checks = {
        "Corporate bonds": "corp_issue_id",
        "Treasuries": "treasury_maturity_date",
        "ABS": "abs_issuer",
        "MBS": "mbs_issuer",
    }

    for name, col in checks.items():
        if col not in merged.columns:
            print(f"{name}: column not available")
            continue

        matched = merged[col].notna().sum()
        total = len(merged)
        print(
            f"{name}: {matched:,}/{total:,} securities matched "
            f"({matched / total:.1%})"
            if total else f"{name}: empty universe"
        )


report_match_rates(bond_reference_df)

In [2]:
import os
import psutil
import pandas as pd

mem = psutil.virtual_memory()

print(f"Total RAM: {mem.total / 1024**3:.2f} GB")
print(f"Available RAM: {mem.available / 1024**3:.2f} GB")
print(f"Pandas version: {pd.__version__}")

Total RAM: 8.00 GB
Available RAM: 1.20 GB
Pandas version: 2.2.3
